# Продвинутые методы проверки гипотез

Это продолжение лекции про статистические гипотезы. Раньше мы научились сравнивать **две** группы (t-тест, Манн-Уитни) и проверять гипотезы о **среднем**. Здесь разберём, что делать, когда групп **больше двух**, когда данные **категориальные** (а не числа), и почему "p < 0.05" — это ещё не вся история.

Логика та же самая, что и раньше: считаем статистику → сравниваем с распределением → получаем p-value → сравниваем с α. Меняются только формулы и вопросы, на которые они отвечают.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
import statsmodels.api as sm
from statsmodels.formula.api import ols
from statsmodels.stats.multicomp import pairwise_tukeyhsd


## 1. Однофакторный дисперсионный анализ (ANOVA)

**Вопрос, на который отвечает ANOVA:** отличается ли среднее хотя бы в одной из **нескольких** групп от остальных?

Можно, конечно, сравнить группы попарно t-тестом. Но если групп 5, попарных сравнений будет 10 — и с каждым тестом растёт шанс случайно поймать "значимость" там, где её нет (об этом — в разделе про поправки на множественные сравнения). ANOVA решает это одним тестом сразу для всех групп.

**Идея метода** — разложить общий разброс данных на две части:
- **межгрупповая дисперсия** — насколько сильно отличаются средние групп друг от друга;
- **внутригрупповая дисперсия** — насколько сильно "шумят" сами данные внутри каждой группы.

Если группы правда разные, межгрупповая дисперсия будет **намного больше** внутригрупповой. Это отношение и есть статистика:

$$
F = \frac{MS_{between}}{MS_{within}}
$$

**H₀:** все средние равны (μ₁ = μ₂ = μ₃ = …). **H₁:** хотя бы одно среднее отличается.

Важно: ANOVA не говорит, *какая именно* группа отличается — только то, что где-то есть различие. Для "какая именно" нужны post-hoc тесты (раздел 2).

In [2]:
# Пример: три способа обучения, баллы за тест
np.random.seed(0)
method_A = np.random.normal(loc=70, scale=5, size=30)
method_B = np.random.normal(loc=75, scale=5, size=30)
method_C = np.random.normal(loc=70, scale=5, size=30)

F_stat, p_value = stats.f_oneway(method_A, method_B, method_C)
print(f"F = {F_stat:.3f}, p-value = {p_value:.4f}")
if p_value < 0.05:
    print("H0 отвергаем: хотя бы одна группа отличается")
else:
    print("Нет оснований считать группы разными")


F = 5.625, p-value = 0.0050
H0 отвергаем: хотя бы одна группа отличается


Раскроем формулу F руками — так понятнее, откуда она берётся.

In [3]:
groups = [method_A, method_B, method_C]
k = len(groups)                          # число групп
n_i = [len(g) for g in groups]           # размер каждой группы
N = sum(n_i)                             # всего наблюдений

grand_mean = np.concatenate(groups).mean()          # общее среднее по всем данным
group_means = [g.mean() for g in groups]             # среднее внутри каждой группы

# межгрупповая изменчивость: насколько средние групп разбросаны вокруг общего среднего
SS_between = sum(n_i[i] * (group_means[i] - grand_mean) ** 2 for i in range(k))

# внутригрупповая изменчивость: насколько точки разбросаны вокруг своего группового среднего
SS_within = sum(((g - g.mean()) ** 2).sum() for g in groups)

df_between = k - 1
df_within = N - k

MS_between = SS_between / df_between
MS_within = SS_within / df_within

F_manual = MS_between / MS_within
print(f"SS_between = {SS_between:.1f}, SS_within = {SS_within:.1f}")
print(f"F (вручную) = {F_manual:.3f}  (совпадает с F выше)")


SS_between = 279.2, SS_within = 2159.0
F (вручную) = 5.625  (совпадает с F выше)


## 2. Post-hoc тесты: какая группа отличается?

ANOVA сказала "где-то есть разница", но не сказала где. Post-hoc тест (Тьюки, Tukey HSD) сравнивает **все пары** групп, но правильно учитывает, что сравнений много — не завышая ложно шанс найти отличие.

Читать результат просто: колонка `reject` — `True` значит пара групп статистически различается.

In [4]:
labels = ["A"] * 30 + ["B"] * 30 + ["C"] * 30
scores = np.concatenate([method_A, method_B, method_C])

tukey = pairwise_tukeyhsd(endog=scores, groups=labels, alpha=0.05)
print(tukey)


Multiple Comparison of Means - Tukey HSD, FWER=0.05
group1 group2 meandiff p-adj   lower  upper  reject
---------------------------------------------------
     A      B   1.3381 0.5536 -1.7289 4.4051  False
     A      C  -2.8829 0.0699 -5.9499 0.1841  False
     B      C   -4.221 0.0042  -7.288 -1.154   True
---------------------------------------------------


Читаем `reject`: пара **B-C** значимо различается (p-adj = 0.004), а пары A-B и A-C — нет, хотя A-C и близко к границе (p-adj = 0.07).

Мы генерировали данные с истинными средними 70/75/70 для A/B/C — то есть на самом деле B отличается от **обеих** групп. Но на конкретной выборке различие A-B "потерялось" из-за случайного шума. Это иллюстрация ошибки II рода из прошлой лекции: реальный эффект есть, но выборки не хватило, чтобы его надёжно поймать.

## 3. Двухфакторный дисперсионный анализ (Two-way ANOVA)

Что если на результат влияют **два** фактора сразу? Например: способ обучения **и** пол ученика. Двухфакторный ANOVA проверяет сразу три гипотезы:

1. влияет ли фактор 1 (способ обучения) сам по себе;
2. влияет ли фактор 2 (пол) сам по себе;
3. есть ли **взаимодействие** — то есть эффект одного фактора зависит от уровня другого (например, метод B работает лучше только для мальчиков).

Каждая гипотеза получает свой F и своё p-value.

In [5]:
np.random.seed(1)
n = 20
data = pd.DataFrame({
    "method": np.repeat(["A", "B"], n * 2),
    "gender": np.tile(np.repeat(["M", "F"], n), 2),
})
# score зависит от метода, слегка от пола, и есть небольшое взаимодействие
base = 70
effect_method = data["method"].map({"A": 0, "B": 4})
effect_gender = data["gender"].map({"M": 0, "F": 1})
interaction = ((data["method"] == "B") & (data["gender"] == "F")).astype(int) * 5
data["score"] = base + effect_method + effect_gender + interaction + np.random.normal(0, 4, len(data))

model = ols("score ~ C(method) * C(gender)", data=data).fit()
anova_table = sm.stats.anova_lm(model, typ=2)
print(anova_table.round(4))


                        sum_sq    df        F  PR(>F)
C(method)            1159.1071   1.0  77.0466  0.0000
C(gender)             261.0696   1.0  17.3535  0.0001
C(method):C(gender)   102.4077   1.0   6.8071  0.0109
Residual             1143.3619  76.0      NaN     NaN


Смотрим на колонку `PR(>F)` — это p-value для каждой строки:
- `C(method)` — значим ли сам способ обучения;
- `C(gender)` — значим ли сам пол;
- `C(method):C(gender)` — значимо ли взаимодействие (эффект метода отличается между M и F).

Маленькое p-value у взаимодействия говорит: нельзя просто сказать "метод B лучше" — надо уточнять, для кого именно.

## 4. Критерий Краскела-Уоллиса — ANOVA без нормальности

ANOVA, как и t-тест, предполагает, что данные внутри групп примерно нормальны. Если это не так (выбросы, сильная асимметрия), используют **непараметрический** аналог — критерий Краскела-Уоллиса. Он сравнивает не сами значения, а их **ранги** (как Манн-Уитни, но для 3+ групп).

In [6]:
np.random.seed(0)
g1 = np.random.exponential(scale=2, size=40)
g2 = np.random.exponential(scale=2, size=40)
g3 = np.random.exponential(scale=2.6, size=40)  # эта группа реально немного другая

H_stat, p_kw = stats.kruskal(g1, g2, g3)
print(f"Kruskal-Wallis: H = {H_stat:.3f}, p-value = {p_kw:.4g}")

# для сравнения — обычная ANOVA на тех же (не нормальных!) данных
F_stat, p_anova = stats.f_oneway(g1, g2, g3)
print(f"ANOVA:          F = {F_stat:.3f}, p-value = {p_anova:.4g}")


Kruskal-Wallis: H = 9.068, p-value = 0.01074
ANOVA:          F = 2.321, p-value = 0.1027


На скошенных данных (экспоненциальное распределение, всегда есть немного "длинного хвоста") эффект в группе 3 реален, но экстремальные значения раздувают внутригрупповую дисперсию — и ANOVA не набирает уверенности (p > 0.05). Краскел-Уоллис работает с рангами, а не с самими значениями, поэтому "хвост" его не сбивает — и он корректно ловит отличие (p < 0.05).

## 5. Хи-квадрат: критерий согласия (goodness of fit)

Всё, что было выше, — про **числовые** данные (средние, дисперсии). А если данные **категориальные** — например, "сколько покупателей выбрали каждый из 3 тарифов"? Здесь работает критерий χ² (хи-квадрат).

**Критерий согласия** проверяет, соответствует ли наблюдаемое распределение по категориям — ожидаемому (теоретическому).

$$
\chi^2 = \sum_{i=1}^{k} \frac{(O_i - E_i)^2}{E_i}
$$

где $O_i$ — наблюдаемая частота в категории $i$, $E_i$ — ожидаемая. Чем сильнее наблюдаемые частоты отличаются от ожидаемых, тем больше χ² и тем меньше p-value.

**H₀:** наблюдаемое распределение совпадает с ожидаемым.

In [7]:
# Пример: ожидаем, что тарифы выбирают поровну (по 100 из 300 клиентов)
observed = np.array([120, 90, 90])
expected = np.array([100, 100, 100])

chi2_stat, p_value = stats.chisquare(f_obs=observed, f_exp=expected)
print(f"chi2 = {chi2_stat:.3f}, p-value = {p_value:.4f}")
if p_value < 0.05:
    print("H0 отвергаем: распределение по тарифам НЕ равномерное")
else:
    print("Нет оснований считать распределение неравномерным")


chi2 = 6.000, p-value = 0.0498
H0 отвергаем: распределение по тарифам НЕ равномерное


## 6. Хи-квадрат: критерий независимости (таблицы сопряжённости)

Другой частый вопрос: **связаны ли две категориальные переменные** между собой? Например: зависит ли выбор тарифа от города клиента?

Строим таблицу сопряжённости (строки — города, столбцы — тарифы) и проверяем H₀: переменные независимы (город не влияет на выбор тарифа).

In [8]:
# строки — город, столбцы — тариф (Basic / Pro / Premium)
table = np.array([
    [50, 30, 20],   # Москва
    [20, 40, 40],   # Питер
])

chi2_stat, p_value, dof, expected = stats.chi2_contingency(table)
print(f"chi2 = {chi2_stat:.3f}, df = {dof}, p-value = {p_value:.5f}")
print("Ожидаемые частоты при независимости:")
print(np.round(expected, 1))
if p_value < 0.05:
    print("H0 отвергаем: выбор тарифа связан с городом")
else:
    print("Нет оснований считать выбор тарифа и город связанными")


chi2 = 20.952, df = 2, p-value = 0.00003
Ожидаемые частоты при независимости:
[[35. 35. 30.]
 [35. 35. 30.]]
H0 отвергаем: выбор тарифа связан с городом


`expected` — это частоты, которые были бы, если бы город и тариф были никак не связаны. Наблюдаемая таблица сильно от них отличается (в Питере заметно чаще берут Premium) — отсюда и маленький p-value.

## 7. Поправка на множественные сравнения

Возвращаемся к проблеме из раздела 2. У каждого теста есть 5% шанс (при α = 0.05) ложно отвергнуть верную H₀ — это заложено в определении α. Но если сделать **много** тестов подряд, шанс хотя бы одной ложной "находки" быстро растёт:

$$
P(\text{хотя бы одна ложная находка}) = 1-(1-\alpha)^m
$$

где $m$ — число тестов. При 20 независимых тестах с α = 0.05 это уже ~64%, а не 5%!

**Поправка Бонферрони** — самый простой способ справиться: делить α на число сравнений $m$, то есть требовать значимости на уровне $\alpha/m$ для каждого отдельного теста.

In [9]:
m = 20
alpha = 0.05

chance_false_positive = 1 - (1 - alpha) ** m
print(f"Шанс хотя бы одной ложной находки среди {m} тестов: {chance_false_positive:.1%}")

alpha_corrected = alpha / m
print(f"Скорректированный порог (Бонферрони): {alpha_corrected:.4f} вместо {alpha}")


Шанс хотя бы одной ложной находки среди 20 тестов: 64.2%
Скорректированный порог (Бонферрони): 0.0025 вместо 0.05


Поправка Бонферрони — довольно строгая (может пропустить реальные эффекты при большом m). На практике часто используют более мягкие варианты (поправка Холма, контроль FDR методом Бенджамини-Хохберга), но идея та же: чем больше тестов, тем строже должен быть порог для каждого отдельного.

## 8. Размер эффекта: p-value — это не всё

Маленькое p-value говорит "эффект, скорее всего, не случаен". Но оно **ничего не говорит о размере** эффекта — а часто именно размер и важен на практике. При очень большой выборке даже крошечная, никому не интересная разница даст p < 0.05.

**Cohen's d** — стандартный размер эффекта для сравнения двух средних: разница средних, выраженная в стандартных отклонениях.

$$
d = \frac{\bar{X_1} - \bar{X_2}}{s_{pooled}}
$$

Условно: d ≈ 0.2 — маленький эффект, d ≈ 0.5 — средний, d ≈ 0.8 и выше — большой (это грубые ориентиры, не строгие границы).

In [10]:
def cohens_d(a, b):
    n1, n2 = len(a), len(b)
    s1, s2 = a.var(ddof=1), b.var(ddof=1)
    s_pooled = np.sqrt(((n1 - 1) * s1 + (n2 - 1) * s2) / (n1 + n2 - 2))
    return (a.mean() - b.mean()) / s_pooled

np.random.seed(3)
# огромная выборка, разница в среднем всего 0.1 -- но p-value всё равно маленький
big_a = np.random.normal(loc=50.0, scale=5, size=50000)
big_b = np.random.normal(loc=50.1, scale=5, size=50000)

t_stat, p_val = stats.ttest_ind(big_a, big_b)
d = cohens_d(big_a, big_b)
print(f"t-test: p-value = {p_val:.4g}  (статистически значимо!)")
print(f"Cohen's d = {d:.4f}  (эффект практически нулевой)")


t-test: p-value = 0.0006236  (статистически значимо!)
Cohen's d = -0.0216  (эффект практически нулевой)


Вывод: **всегда** смотрите на размер эффекта вместе с p-value. "Статистически значимо" ≠ "важно на практике".

# Итоговая шпаргалка: какой тест выбрать

| Вопрос | Числовые данные, 2 группы | Числовые данные, 3+ группы | Категориальные данные |
|---|---|---|---|
| Данные нормальны | t-тест | ANOVA | — |
| Данные не нормальны / выбросы | Манн-Уитни | Краскел-Уоллис | — |
| Соответствие теоретическому распределению | — | — | χ² согласия |
| Связь двух категориальных переменных | — | — | χ² независимости |
| "Какая именно группа отличается" после ANOVA | — | Tukey HSD (post-hoc) | — |

И не забывать: при множестве тестов — поправка (Бонферрони/Холм), и рядом с p-value всегда смотреть на размер эффекта (Cohen's d).

# Что дальше?

1) Регрессионный анализ (линейная регрессия как обобщение сравнения средних)
2) ANCOVA (ANOVA с учётом непрерывной ковариаты)
3) Бутстрэп и доверительные интервалы без предположений о распределении
4) Байесовский подход к проверке гипотез